# 免费 Colab：ForTy GPU 管线检查

在 https://colab.research.google.com 上传此 Notebook，登录自己的 Google
账号，在“运行时 → 更改运行时类型”选择可用的 GPU，然后按顺序运行。
免费 GPU 的供应、配额和运行时长不固定；此 Notebook 不创建收费 VM、不开通付费套餐。

默认使用完整模型，ComboLoss 和普通交叉熵**各训练两轮**，每个划分一个分片、
一个种子、batch size 1。这是小规模真实数据流程检查，分数仅作诊断，
不能据此判断模型优劣。不要根据测试指标调整模型或训练设置。

准备好同一份交付中的 `publication-recovery.zip` 或
`geospatial-mamba-deforestation.bundle`，运行上传单元格时选择它。
Notebook 固定到已验证的源码提交，不依赖尚未推送的 GitHub 修改。
此入口尚未在真实 Colab GPU 上执行；下面的运行检查决定实际可用性。

In [ ]:
import datetime
import io
import json
import os
from pathlib import Path
import signal
import subprocess
import sys
import uuid
import zipfile

SOURCE_COMMIT = "ae6a123d0469a2950ce6b9733cf7902b14cb7f5c"
SOURCE_BRANCH = "codex/reproducible-evaluation-performance"
DATA_ROOT = "gs://forest_typology/forty_v1/1.0.0"
TRAIN_SHARDS = 1
VAL_SHARDS = 1
TEST_SHARDS = 1
EPOCHS = 2
BATCH_SIZE = 1
SEEDS = [42]
SAVE_TO_DRIVE = False

WORKSPACE = Path("/content/geospatial-mamba")
REPO = WORKSPACE / "source"
WORKSPACE.mkdir(parents=True, exist_ok=True)
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    output_base = Path("/content/drive/MyDrive/geospatial-mamba")
else:
    output_base = WORKSPACE / "artifacts"
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
SESSION_ROOT = output_base / ("colab-" + stamp + "-" + uuid.uuid4().hex[:8])
SESSION_ROOT.mkdir(parents=True, exist_ok=False)
RUN_ROOT = SESSION_ROOT / "experiment"
os.environ.setdefault("MPLCONFIGDIR", str(WORKSPACE / "cache/matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(WORKSPACE / "cache"))
print("Artifacts:", SESSION_ROOT)
print("Each arm:", EPOCHS, "epochs; train/val/test shards:",
      TRAIN_SHARDS, VAL_SHARDS, TEST_SHARDS)

`SAVE_TO_DRIVE=True` 会由你授权挂载自己的 Drive，并从一开始把输出写在那里。
默认 `/content` 是临时空间，运行结束后应立即下载结果。免费运行时被强制回收时，
导出代码可能没有机会执行；Drive 也只能保留已经成功写入的文件。
当前训练不支持恢复优化器状态，中断后请使用新目录重跑。
再次运行配置单元格会创建新的实验目录，保留旧目录。

In [ ]:
def bundle_bytes(filename, payload):
    if filename.endswith(".bundle"):
        return payload
    if not filename.endswith(".zip"):
        raise ValueError("请选择交付的 .zip 恢复包或 .bundle 文件")
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        members = [item for item in archive.infolist()
                   if item.filename == "geospatial-mamba-deforestation.bundle"]
        if len(members) != 1:
            raise ValueError("恢复包中必须恰好有一个项目 Git bundle")
        if members[0].file_size > 64 * 1024 * 1024:
            raise ValueError("Git bundle 超过此入口的大小限制")
        return archive.read(members[0])

if not REPO.exists():
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("请每次只上传一个恢复包")
    filename, payload = next(iter(uploaded.items()))
    bundle = WORKSPACE / ("source-" + uuid.uuid4().hex + ".bundle")
    bundle.write_bytes(bundle_bytes(filename, payload))
    subprocess.run(["git", "clone", "--branch", SOURCE_BRANCH,
                    str(bundle), str(REPO)], check=True)
    subprocess.run(["git", "bundle", "verify", str(bundle)], cwd=REPO, check=True)
    subprocess.run(["git", "checkout", "--detach", SOURCE_COMMIT],
                   cwd=REPO, check=True)
head = subprocess.check_output(["git", "rev-parse", "HEAD"],
                               cwd=REPO, text=True).strip()
dirty = subprocess.check_output(["git", "status", "--porcelain"],
                                cwd=REPO, text=True)
if head != SOURCE_COMMIT or dirty:
    raise RuntimeError("源码提交不符或工作区有修改；请检查后使用新的工作目录")
(SESSION_ROOT / "source_commit.txt").write_text(head + "\n")
print("Source:", head)

In [ ]:
# Reuse the TensorFlow/CUDA stack provided by the Colab runtime.
probe_code = (
    "import json, tensorflow as tf; "
    "print(json.dumps({'version': tf.__version__, "
    "'schemes': tf.io.gfile.get_registered_schemes()}))"
)
probe = subprocess.check_output([sys.executable, "-c", probe_code], text=True)
runtime = json.loads(probe.strip().splitlines()[-1])
major_minor = tuple(int(part) for part in runtime["version"].split(".")[:2])
if major_minor < (2, 15):
    raise RuntimeError("此入口需要 TensorFlow 2.15 或更新版本的 Colab 运行时")
if "gs" not in runtime["schemes"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps",
                    "tensorflow-io-gcs-filesystem"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(REPO)],
               check=True)
packages = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
(SESSION_ROOT / "packages.txt").write_text(packages)
print("Colab TensorFlow:", runtime["version"])

In [ ]:
# Require real GPU convolution gradients and GCS registration before data IO.
subprocess.run([sys.executable, str(REPO / "cloud/check_runtime.py"),
                "--output", str(SESSION_ROOT / "runtime-readiness.json")],
               cwd=REPO, check=True)
readiness = json.loads((SESSION_ROOT / "runtime-readiness.json").read_text())
print(json.dumps(readiness, indent=2))

以下先生成并显示计划，再执行。预检会读取 train/validation/test 各一个样本检查格式；
这不是全数据或地理独立性验证。两组训练都完成后才计算测试评估指标。
设置应在这里冻结；本次小规模测试分数不能用来选择后续正式实验的超参数。
诊断文件在会话目录中，runner 使用其下全新的 `experiment` 目录。

In [ ]:
def experiment_command(execute=False):
    command = [sys.executable, "-m", "mamba_forest.experiment",
               "--run-root", str(RUN_ROOT), "--data-root", DATA_ROOT,
               "--train-shards", str(TRAIN_SHARDS), "--val-shards", str(VAL_SHARDS),
               "--test-shards", str(TEST_SHARDS), "--epochs", str(EPOCHS),
               "--batch-size", str(BATCH_SIZE), "--seeds", *map(str, SEEDS)]
    if execute:
        command.append("--execute")
    return command

subprocess.run(experiment_command(), cwd=REPO, check=True)
plan = json.loads((RUN_ROOT / "plan.json").read_text())
print(json.dumps(plan["config"], indent=2))
print("Stages:", [step["id"] for step in plan["steps"]])

In [ ]:
def archive_session():
    destination = SESSION_ROOT.parent / (SESSION_ROOT.name + ".zip")
    with zipfile.ZipFile(destination, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(SESSION_ROOT.rglob("*")):
            if path.is_file() and not path.is_symlink():
                archive.write(path, path.relative_to(SESSION_ROOT))
    print("Archive:", destination)
    return destination

def execute_with_cleanup(argv, cwd):
    process = subprocess.Popen(argv, cwd=cwd, start_new_session=True)
    try:
        return_code = process.wait()
    except BaseException:
        try:
            try:
                os.killpg(process.pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                pass
        finally:
            # A stopped runner can leave a child in its process group.
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            process.wait()
        raise
    if return_code:
        raise subprocess.CalledProcessError(return_code, argv)

In [ ]:
try:
    execute_with_cleanup(experiment_command(execute=True), cwd=REPO)
    report = json.loads((RUN_ROOT / "comparison.json").read_text())
    print("Configured data source:", DATA_ROOT)
    print("Small-source pipeline scores only; no real-data model ranking is established.")
    print(json.dumps(report["summary"], indent=2))
finally:
    ARCHIVE = archive_session()

成功或普通错误后都会尝试打包权重、日志、配置与指标。点击“停止”时会先清理训练进程；
整个运行时被回收时无法保证打包。修复失败后需要新实验目录，不能复用已执行目录。
下方单元格可重新打包并下载现存文件；Drive 模式还可以在自己的 Drive 中查看输出。
正式三种子、全测试集对照另见仓库的 `docs/gpu-validation.md`，免费配额未必足够。

In [ ]:
from google.colab import files
ARCHIVE = archive_session()
files.download(str(ARCHIVE))